# Part 1: direct GMM reason prediction

This notebook tests a **single GMM-based predictor**. It does not assign pseudo-labels to unlabeled training trials and then fit another classifier.

The GMM groups all training embeddings. The originally visible reason labels tell us the rate of each reason in each mixture component. For a new trial, combine the component reason rates using its GMM membership probabilities:

`reason_probability = GMM membership probabilities @ component reason rates`

Reason rates are estimated only from originally labeled training rows. Reasons are independent, so the five probabilities need not sum to one. We score on validation before generating a distinct Kaggle submission. IDs are only carried to the output file, not used as model features.

In [23]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import KernelPCA
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

## Load Part 1 data

The kernel will look for `part1-data` in the current directory and its ancestors, then use the project's local path as a fallback.

In [24]:
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))
PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data. Kernel working directory: {cwd}")

ATTEMPT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"
DATA_DIR = PROJECT_DIR / "part1-data"

In [25]:
part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")

print("Training:", part1_train.shape)
print("Validation:", part1_val.shape)
print("Test:", part1_test.shape)

Training: (1171, 166)
Validation: (78, 166)
Test: (312, 161)


## Configure the experiment

The first comparison uses the same RBF kernel PCA representation and GMM settings as the earlier GMM experiment. It sweeps 2–30 components. Change one setting at a time so the results are easier to interpret.

In [26]:
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDINGS = [f"emb_{i}" for i in range(150)]

KERNEL_PCA_COMPONENTS = 50
KERNEL = "rbf"
KERNEL_GAMMA = 1 / 75
KERNEL_PCA_SEED = 42

GMM_COVARIANCE_TYPE = "diag"
GMM_N_INIT = 5
GMM_MAX_ITER = 300
GMM_REG_COVAR = 1e-4
GMM_SEED = 42
COMPONENT_COUNTS = range(2, 31)
REASON_THRESHOLD = 0.50

## Scale and transform embeddings

Fit scaling and Kernel PCA using training embeddings only. Validation and test rows are transformed by those fitted steps; neither split affects fitting.

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(part1_train[EMBEDDINGS])
X_val_scaled = scaler.transform(part1_val[EMBEDDINGS])
X_test_scaled = scaler.transform(part1_test[EMBEDDINGS])

kernel_pca = KernelPCA(
    n_components=KERNEL_PCA_COMPONENTS,
    kernel=KERNEL,
    gamma=KERNEL_GAMMA,
    eigen_solver="arpack",
    random_state=KERNEL_PCA_SEED,
)
X_train_gmm = kernel_pca.fit_transform(X_train_scaled)
X_val_gmm = kernel_pca.transform(X_val_scaled)
X_test_gmm = kernel_pca.transform(X_test_scaled)

print("GMM input shapes:", X_train_gmm.shape, X_val_gmm.shape, X_test_gmm.shape)
print(f"Kernel: {KERNEL}; gamma: {KERNEL_GAMMA:g}")

GMM input shapes: (1171, 50) (78, 50) (312, 50)
Kernel: rbf; gamma: 0.0133333


## Estimate reason rates for the components

For each GMM component, average the **originally known training labels**, weighted by the GMM membership probabilities. A trial can contribute partly to several components. The returned support is the summed membership weight from known trials, not a count of integer-assigned trials.

In [28]:
def component_reason_rates(model, X_train, train_df):
    """Estimate five reason rates per component from originally labeled rows."""
    if len(X_train) != len(train_df):
        raise ValueError("Training embeddings and labels must have matching row order.")

    known_mask = train_df[REASONS].isin([0, 1]).all(axis=1).to_numpy()
    if not known_mask.any():
        raise ValueError("No originally labeled training trials are available.")

    responsibilities = model.predict_proba(X_train)
    y_known = train_df.loc[known_mask, REASONS].to_numpy(dtype=float)
    known_responsibilities = responsibilities[known_mask]
    effective_support = known_responsibilities.sum(axis=0)
    overall_rates = y_known.mean(axis=0)

    weighted_positive_mass = known_responsibilities.T @ y_known
    rates = np.divide(
        weighted_positive_mass,
        effective_support[:, None],
        out=np.tile(overall_rates, (model.n_components, 1)),
        where=effective_support[:, None] > 1e-12,
    )
    rates = np.clip(rates, 0.0, 1.0)
    return rates, effective_support

In [29]:
def reason_probabilities(model, X, component_rates):
    """Predict five reason probabilities directly from one fitted GMM."""
    return np.clip(model.predict_proba(X) @ component_rates, 0.0, 1.0)

## Convert reason probabilities to labels and score validation

Apply the threshold independently to each reason. If all five probabilities fall below threshold, assign the single most probable reason. This fallback keeps every trial's Kaggle `Predicted` field nonempty; the table reports how often it was needed on validation.

In [30]:
def probabilities_to_reason_labels(probabilities, threshold=REASON_THRESHOLD):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    fallback_rows = ~predictions.any(axis=1)
    if fallback_rows.any():
        best_reason = np.asarray(probabilities)[fallback_rows].argmax(axis=1)
        predictions[np.flatnonzero(fallback_rows), best_reason] = 1
    return predictions, fallback_rows


def evaluate_reason_predictions(y_true, y_pred):
    per_reason = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {
        "Sample-wise F1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        **dict(zip(REASONS, per_reason)),
    }

## Run the direct GMM component sweep

Each component count fits one GMM on all training embeddings. Its soft memberships and the originally labeled training rows determine the component reason rates. The same fitted GMM predicts validation reason probabilities directly; no pseudo-labeled training set or second classifier is fitted.

In [31]:
y_val = part1_val[REASONS].to_numpy(dtype=int)
direct_gmm_results = {}

for n_components in COMPONENT_COUNTS:
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type=GMM_COVARIANCE_TYPE,
        n_init=GMM_N_INIT,
        max_iter=GMM_MAX_ITER,
        reg_covar=GMM_REG_COVAR,
        random_state=GMM_SEED,
    )
    gmm.fit(X_train_gmm)

    rates, effective_support = component_reason_rates(gmm, X_train_gmm, part1_train)
    val_probabilities = reason_probabilities(gmm, X_val_gmm, rates)
    val_predictions, val_fallback_rows = probabilities_to_reason_labels(
        val_probabilities, threshold=REASON_THRESHOLD
    )
    metrics = evaluate_reason_predictions(y_val, val_predictions)

    direct_gmm_results[n_components] = {
        "model": gmm,
        "component_reason_rates": rates,
        "effective_labeled_support": effective_support,
        "val_probabilities": val_probabilities,
        "val_predictions": val_predictions,
        "validation_metrics": metrics,
        "validation_fallback_count": int(val_fallback_rows.sum()),
        "converged": bool(gmm.converged_),
        "iterations": int(gmm.n_iter_),
    }

## Compare all component counts

The competition metric is sample-wise F1. Also inspect macro and per-reason F1, effective component support, convergence, and fallback counts before choosing a model. The direct predictor does not pseudo-label training rows, so “unresolved training percentage” does not apply; validation fallback count shows how many rows needed the nonempty-label rule.

In [32]:
score_rows = []
for n_components, run in direct_gmm_results.items():
    metrics = run["validation_metrics"]
    score_rows.append({
        "Components": n_components,
        **metrics,
        "Min effective labeled support": run["effective_labeled_support"].min(),
        "Validation fallback rows": run["validation_fallback_count"],
        "Converged": run["converged"],
        "EM iterations": run["iterations"],
    })

direct_gmm_scores = pd.DataFrame(score_rows).set_index("Components").sort_index()
display(direct_gmm_scores.round(4))
best_components = int(direct_gmm_scores["Sample-wise F1"].idxmax())
print("Best validation sample-wise F1 component count:", best_components)

,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Min effective labeled support,Validation fallback rows,Converged,EM iterations
Components,,,,,,,,,,,
2,0.6239,0.1562,0.0000,0.0,0.0000,0.7812,0.0000,67.9126,0,True,13
3,0.6239,0.1562,0.0000,0.0,0.0000,0.7812,0.0000,23.3339,24,True,9
4,0.6239,0.1562,0.0000,0.0,0.0000,0.7812,0.0000,24.0000,26,True,25
5,0.5470,0.2271,0.0000,0.0,0.0000,0.8090,0.3265,7.0000,40,True,18
6,0.5470,0.2315,0.0000,0.0,0.0000,0.8046,0.3529,2.0000,41,True,16
7,0.6239,0.1562,0.0000,0.0,0.0000,0.7812,0.0000,0.0000,51,True,28
8,0.5855,0.2395,0.0000,0.0,0.0000,0.8571,0.3404,0.0000,38,True,19
9,0.6111,0.2472,0.0000,0.0,0.0000,0.8723,0.3636,0.0000,35,True,23
10,0.5983,0.2375,0.0000,0.0,0.0000,0.8542,0.3333,0.0000,33,True,22


Best validation sample-wise F1 component count: 29


## Export one chosen component count for Kaggle

After reviewing validation results, choose a component count below. This cell uses the same GMM and component reason rates from the sweep to predict test rows directly. It writes `ID,Predicted` to a GMM-specific filename and does not upload anything.

In [33]:
GMM_EXPORT_COMPONENTS = 21  # Change after reviewing the validation table.

export_run = direct_gmm_results[GMM_EXPORT_COMPONENTS]
test_probabilities = reason_probabilities(
    export_run["model"], X_test_gmm, export_run["component_reason_rates"]
)
test_predictions, test_fallback_rows = probabilities_to_reason_labels(
    test_probabilities, threshold=REASON_THRESHOLD
)

submission = part1_test[["ID"]].copy()
submission["Predicted"] = [
    " ".join(reason for reason, value in zip(REASONS, row) if value)
    for row in test_predictions
]
output_path = ATTEMPT_DIR / "part1_gmm_direct_kernel_pca_submission.csv"
submission.to_csv(output_path, index=False)
print(f"Saved {len(submission)} rows to {output_path}")
print("Rows using highest-probability fallback:", int(test_fallback_rows.sum()))
submission.head()

Saved 312 rows to /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1_gmm_direct_kernel_pca_submission.csv
Rows using highest-probability fallback: 200


,ID,Predicted
0,TR00756,Safety
1,TR05267,Other
2,TR10354,Other
3,TR01930,Recruitment
4,TR01046,Other
